# 12. Batch and Async Processing

**Practical use case:** Process many independent LLM requests efficiently.

This trainer-ready notebook contains explanation, live `langchain_openai` code, validation guidance and exercises. It makes real API calls and may incur charges.

## 1. Business problem

**Scenario:** Process several independent business requests efficiently.

The goal is to convert an unstructured language task into a repeatable workflow that can be demonstrated, tested and later integrated into an application.

## 2. Solution workflow

1. Prepare or load the input.
2. Define the model and important parameters.
3. Construct a precise prompt or schema.
4. Invoke the model.
5. inspect and validate the response.
6. Save or pass the result to the next application step.

### 1. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

### 2. Process multiple prompts as a batch

This cell submits multiple independent prompts as a batch with controlled concurrency.

**Expected result:** One response is printed for every input item while preserving result order. Read the output before continuing to the next cell.

In [ ]:
llm = ChatOpenAI(model=MODEL_NAME, temperature=0, max_retries=2)
requests = [
    "Classify in one word: The service was excellent.",
    "Classify in one word: Delivery was delayed.",
    "Classify in one word: The product is acceptable.",
]
responses = llm.batch(requests, config={"max_concurrency": 3})
for request, response in zip(requests, responses):
    print(request, "->", response.content)

### 3. Run the asynchronous model call

This cell uses the asynchronous LangChain interface so independent requests do not block one another.

**Expected result:** Responses are returned after the awaited operation completes. Read the output before continuing to the next cell.

In [ ]:
# Run this cell in Jupyter to demonstrate native asynchronous calls.
async_responses = await llm.abatch(requests, config={"max_concurrency": 3})
for response in async_responses:
    print(response.content)

## Expected result

The model should return an output that follows the requested scope and format. During training, compare the actual response with the requirement instead of assuming that a fluent response is correct.

## Parameter experiment

Run the example once with `temperature=0` and once with a higher supported temperature. Compare consistency, wording and creativity. Keep other inputs unchanged so the comparison is meaningful.

## Output validation

Check that the response:

- follows the requested format;
- contains no invented facts;
- preserves important names and numbers;
- is relevant to the business question;
- can be safely used by the next system.

## Error handling

Production code should handle missing API keys, authentication failures, rate limits, timeouts, invalid structured output and temporary service errors. Use limited retries and provide a clear fallback message.